# Section 4: Sarcasm Explanation & Error Analysis
**Author:** Shoeb Iqbal  
**Task:** Extract erroneous predictions from the best LLM/adapter, build a few-shot prompt from 4 explainable examples, test on the remaining 6, and comment on improvement.

> ⚠️ **TODO:** Plug in your teammates' model/predictions once they're done. Marked with `# TODO` throughout.

## 0. Setup & Imports

In [ ]:
!pip install transformers torch peft -q
!pip install pandas scikit-learn matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

---
## Step 1: Load Model Predictions

> ⚠️ **TODO:** Replace the placeholder below with the actual predictions from Berkin/Shoeb's best LoRA model (Section 2.3).  
> Expected columns: `text`, `true_label`, `predicted_label`, `variety` (en-AU / en-GB / en-US)

In [ ]:

df_preds = pd.DataFrame({
    'text': [
        "Oh great, another Monday.",
        "I just love being stuck in traffic.",
        "The weather is absolutely beautiful today.",
        "Yeah because that always works out so well.",
        "Wow, what a surprise, he was late again.",
        "Sure, because I have nothing better to do.",
        "This is the best day of my life.",
        "Oh fantastic, the printer is broken again.",
        "Nothing like cold coffee to start the morning.",
        "What a wonderful experience that was."
    ],
    'true_label':      [1, 1, 0, 1, 1, 1, 0, 1, 1, 0],
    'predicted_label': [0, 0, 0, 0, 0, 0, 1, 0, 0, 1],
    'variety': ['en-AU','en-GB','en-US','en-AU','en-GB','en-US','en-AU','en-GB','en-US','en-AU']
})
print(f"Total predictions loaded: {len(df_preds)}")
df_preds.head()

---
## Step 2: Extract 10 Erroneous Predictions

In [ ]:

df_errors = df_preds[df_preds['true_label'] != df_preds['predicted_label']].copy()
df_errors['error_type'] = df_errors.apply(
    lambda r: 'False Negative (missed sarcasm)' if r['true_label'] == 1 else 'False Positive (wrongly flagged)',
    axis=1
)

print(f"Total errors found: {len(df_errors)}")


df_10 = df_errors.sample(min(10, len(df_errors)), random_state=42).reset_index(drop=True)

if len(df_10) < 10:
    print(f"⚠️  Only {len(df_10)} errors found — adjust model/dataset source.")

df_10[['text', 'true_label', 'predicted_label', 'error_type', 'variety']]

---
## Step 3: Choose 4 Examples & Write Explanations

> ⚠️ **TODO:** After running Step 2 and seeing the real errors, **manually pick 4** that you can clearly explain. Fill in the `explanation` field for each below.

In [ ]:

chosen_indices = [0, 1, 2, 3]
df_4 = df_10.iloc[chosen_indices].copy().reset_index(drop=True)


explanations = [
    "The phrase 'Oh great, another Monday' uses exaggerated positivity ('great') to express frustration. "
    "The model missed the implicit negativity associated with Monday mornings.",

    "'I just love being stuck in traffic' is clearly sarcastic — no one loves traffic. "
    "The hyperbolic verb 'love' applied to an universally disliked situation is a classic sarcasm marker.",

    "'Yeah because that always works out so well' uses heavy irony. The word 'always' and the tone "
    "imply the opposite of what is literally said. The model failed to pick up on the implied negative history.",

    "'Wow, what a surprise, he was late again' — 'what a surprise' is sarcastic because the event "
    "(being late again) is clearly not surprising. The word 'again' reinforces the irony."
]

df_4['explanation'] = explanations
df_4[['text', 'true_label', 'predicted_label', 'explanation']]

---
## Step 4: Build the Few-Shot Prompt

We construct a single prompt using the 4 chosen examples with their correct labels and explanations.

In [ ]:
def build_few_shot_prompt(df_4_examples, test_text):
    """
    Builds a few-shot prompt using 4 labelled + explained examples,
    then appends a test instance for classification.
    """
    label_map = {1: 'Sarcastic', 0: 'Not Sarcastic'}

    prompt = (
        "You are an expert in detecting sarcasm in text. "
        "Below are examples of sarcastic and non-sarcastic sentences with explanations. "
        "Use these to classify the final sentence.\n\n"
    )

    for i, row in df_4_examples.iterrows():
        prompt += f"Example {i+1}:\n"
        prompt += f"Text: \"{row['text']}\"\n"
        prompt += f"Label: {label_map[row['true_label']]}\n"
        prompt += f"Explanation: {row['explanation']}\n\n"

    prompt += "---\n"
    prompt += f"Now classify the following:\n"
    prompt += f"Text: \"{test_text}\"\n"
    prompt += "Respond with ONLY 'Sarcastic' or 'Not Sarcastic', followed by a one-sentence explanation."

    return prompt

# Preview the prompt with the first of the remaining 6 examples
df_6 = df_10.drop(df_10.index[chosen_indices]).reset_index(drop=True)
sample_prompt = build_few_shot_prompt(df_4, df_6.iloc[0]['text'])
print(sample_prompt)

---
## Step 5: Test the Remaining 6 Examples with the Few-Shot Prompt

> ⚠️ **TODO:** Set `MODEL_PATH` to the path of Berkin/Shoeb's saved LoRA model from Section 2.3.
> The base model should match whatever they used (e.g. `roberta-base` or `xlm-roberta-base`).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

BASE_MODEL = "roberta-base"
LORA_ADAPTER_PATH = "path/to/lora-adapter"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
model = model.to(device)
model.eval()


ID2LABEL = {0: 'Not Sarcastic', 1: 'Sarcastic'}

def query_llm(prompt):
    """
    Runs the LoRA-adapted model on a few-shot prompt.
    The full prompt (with 4 examples + test text) is tokenised and classified.
    Returns a string: 'Sarcastic' or 'Not Sarcastic'.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    pred_id = torch.argmax(outputs.logits, dim=-1).item()
    label = ID2LABEL[pred_id]

    return label

print("LoRA model loaded and ready.")

In [ ]:
results = []

for _, row in df_6.iterrows():
    prompt = build_few_shot_prompt(df_4, row['text'])
    response = query_llm(prompt)

    # Parse prediction from response
    if response == 'Sarcastic':
        few_shot_pred = 1
    elif response == 'Not Sarcastic':
        few_shot_pred = 0
    else:
        few_shot_pred = -1  # unexpected output — check ID2LABEL mapping

    results.append({
        'text': row['text'],
        'true_label': row['true_label'],
        'original_pred': row['predicted_label'],
        'few_shot_pred': few_shot_pred,
        'llm_response': response,
        'variety': row['variety'],
        'original_correct': row['predicted_label'] == row['true_label'],
        'few_shot_correct': few_shot_pred == row['true_label']
    })

df_results = pd.DataFrame(results)
df_results[['text', 'true_label', 'original_pred', 'few_shot_pred', 'few_shot_correct', 'llm_response']]

---
## Step 6: Analysis — Did the Few-Shot Prompt Help?

In [ ]:
label_map = {1: 'Sarcastic', 0: 'Not Sarcastic'}

original_correct = df_results['original_correct'].sum()
few_shot_correct = df_results['few_shot_correct'].sum()
total = len(df_results)

print("=" * 50)
print("RESULTS SUMMARY (on the 6 test examples)")
print("=" * 50)
print(f"Original model correct:    {original_correct}/{total}")
print(f"Few-shot prompt correct:   {few_shot_correct}/{total}")
print()

# Per-example breakdown
print("Per-example breakdown:")
for _, row in df_results.iterrows():
    improved = '✅ IMPROVED' if (not row['original_correct'] and row['few_shot_correct']) else \
               '❌ REGRESSED' if (row['original_correct'] and not row['few_shot_correct']) else \
               '➡️  NO CHANGE'
    print(f"  [{improved}] \"{row['text'][:60]}...\"")
    print(f"    True: {label_map[row['true_label']]} | "
          f"Original: {label_map[row['original_pred']]} | "
          f"Few-shot: {label_map.get(row['few_shot_pred'], 'PARSE ERROR')}")

In [ ]:
# Visualise improvement
import matplotlib.pyplot as plt

categories = ['Original Model', 'Few-Shot Prompt']
correct_counts = [original_correct, few_shot_correct]
wrong_counts = [total - original_correct, total - few_shot_correct]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - width/2, correct_counts, width, label='Correct', color='#4CAF50')
bars2 = ax.bar(x + width/2, wrong_counts, width, label='Wrong', color='#F44336')

ax.set_ylabel('Number of Examples')
ax.set_title('Original Model vs Few-Shot Prompt (6 test examples)')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, total + 1)

for bar in bars1 + bars2:
    ax.annotate(f'{int(bar.get_height())}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha='center')

plt.tight_layout()
plt.savefig('few_shot_comparison.png', dpi=150)
plt.show()
print("Chart saved as few_shot_comparison.png")

---
## Step 7: Written Commentary

> ⚠️ **TODO:** Fill this in once you have real results. The template below gives you the structure.

### Commentary on Few-Shot Prompt Effectiveness

**Overview of errors extracted:**  
We extracted 10 erroneous predictions from the best-performing LoRA-adapted model (Section 2.3). The errors consisted of [X] False Negatives (missed sarcasm) and [Y] False Positives (wrongly flagged as sarcastic), drawn from [varieties].

**Why the 4 examples were chosen:**  
The 4 examples selected for the few-shot prompt were chosen because they represent clear and explainable cases of sarcasm — where the irony stems from [e.g. hyperbolic praise / implied frustration / contradiction with context]. These examples allowed us to write concise, generalizable explanations that the LLM could learn from.

**Results on the remaining 6 examples:**  
The few-shot prompt correctly classified [X/6] examples, compared to [Y/6] by the original model.  
- Examples that **improved**: [list them] — the LLM successfully leveraged the in-context examples to pick up on sarcasm cues it originally missed.
- Examples that **did not improve**: [list them] — these likely involved subtler or more dialect-specific sarcasm that even with explanation the LLM could not resolve.

**Key observations:**  
- The few-shot approach was most effective for [type of sarcasm e.g. hyperbolic expressions].
- It struggled with [e.g. en-AU specific cultural references / deadpan sarcasm without explicit markers].
- Overall, the few-shot prompt [did / did not] provide a meaningful improvement, suggesting that [brief conclusion about LLM sarcasm understanding vs fine-tuned models].